In [1]:
"""
Step 4-0: build a fresh daily burnout time series from the raw post-level BAT
results, for the event-study analysis. Independent of q1_daily_series.csv --
that file was built for the Granger test with a specific CVE-filtering
condition and has nothing to do with this analysis, which only needs burnout
volume (no CVE data at all).

Reads bat_posts_results_with_dates.csv (post-level). Aggregates to one row per
calendar day: total post volume + counts at each BAT-score threshold. Also
picks up individual construct columns (EX/EMO/COG/MD) if your file has them,
so later event-study runs can look at construct-specific patterns -- e.g.
whether MD spikes harder around events than EX does -- not just the aggregate
score.

This is the only script in the event-study pipeline that touches the raw file.
Everything downstream reads the small daily CSV this saves.
"""

import pandas as pd
import numpy as np
import os

# ----------------------------------------------------------------------
# CONFIG -- adjust these if your column names differ. Run once, check the
# printed column list and date range below, and fix before trusting the output.
# ----------------------------------------------------------------------
BAT_CSV = "/Users/nadia/Desktop/redditRun_june/bat_posts_results_with_dates_v2.csv"
OUT_DIR = "/Users/nadia/Desktop/redditRun_june/event_study_v2/"
os.makedirs(OUT_DIR, exist_ok=True)

DATE_COL = "created_date"    # column with the post's date/timestamp
BAT_SCORE_COL = "bat_score"  # 0-4 integer, count of constructs detected

# If your file mixes posts and comments (has both post_id and comment_id
# columns), set this to the value in row_type that means "this row is a post"
# so comments don't get double-counted alongside posts. Set to None to include
# every row as-is (not recommended once you've checked row_type's values).
ROW_TYPE_FILTER = None       # e.g. "post" -- check the printed value_counts below first

# Individual construct flag columns, if they exist in your file. The script
# auto-detects which of these (if any) are actually present -- nothing breaks
# if none of them match, it just falls back to aggregate bat_score only.
CONSTRUCT_COL_CANDIDATES = ["EX", "EMO", "COG", "MD", "ex", "emo", "cog", "md"]


def to_positive_flag(series):
    """Robustly convert a construct column into a boolean 'construct detected'
    indicator, whatever format it's stored in: real booleans, 0/1 integers, or
    string labels like 'YES'/'NO', 'Y'/'N', 'true'/'false'."""
    if series.dtype == bool:
        return series.fillna(False)
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0) >= 1
    normalized = series.astype(str).str.strip().str.lower()
    positive_values = {"yes", "y", "true", "1"}
    return normalized.isin(positive_values)


def main():
    if not os.path.exists(BAT_CSV):
        print(f"Could not find {BAT_CSV}.")
        return

    print(f"Loading {BAT_CSV} ...")
    df = pd.read_csv(BAT_CSV)
    print(f"Loaded {len(df)} posts")
    print(f"Columns found: {list(df.columns)}")

    if "row_type" in df.columns:
        print(f"\nrow_type value counts:\n{df['row_type'].value_counts().to_string()}")
        if ROW_TYPE_FILTER is not None:
            before = len(df)
            df = df[df["row_type"] == ROW_TYPE_FILTER].copy()
            print(f"Filtered to row_type == '{ROW_TYPE_FILTER}': {before} -> {len(df)} rows")
        else:
            print("ROW_TYPE_FILTER is None -- including ALL rows (posts + comments) as-is.\n"
                  "If that's not what you want, set ROW_TYPE_FILTER above to the value that\n"
                  "means 'post' and rerun.")

    if DATE_COL not in df.columns:
        print(f"\nERROR: expected a date column named '{DATE_COL}' but it's not in this file.")
        print("Update DATE_COL at the top of this script to match your actual column name, then rerun.")
        return
    if BAT_SCORE_COL not in df.columns:
        print(f"\nERROR: expected a BAT score column named '{BAT_SCORE_COL}' but it's not in this file.")
        print("Update BAT_SCORE_COL at the top of this script to match your actual column name, then rerun.")
        return

    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    n_bad_dates = df[DATE_COL].isna().sum()
    if n_bad_dates > 0:
        print(f"  WARNING: {n_bad_dates} rows had unparseable dates and will be dropped")
    df = df.dropna(subset=[DATE_COL])
    df["_day"] = df[DATE_COL].dt.normalize()  # collapse timestamp to calendar day

    present_constructs = [c for c in CONSTRUCT_COL_CANDIDATES if c in df.columns]
    if present_constructs:
        print(f"  Found construct columns: {present_constructs}")
        sample_col = present_constructs[0]
        print(f"  Sample values in '{sample_col}': {df[sample_col].dropna().unique()[:10]}")
    else:
        print("  No individual construct columns (EX/EMO/COG/MD) found -- "
              "daily series will only have aggregate bat_score thresholds. "
              "If your file does have per-construct flags under different names, "
              "add them to CONSTRUCT_COL_CANDIDATES above and rerun.")

    df[BAT_SCORE_COL] = pd.to_numeric(df[BAT_SCORE_COL], errors="coerce")
    n_bad_scores = df[BAT_SCORE_COL].isna().sum()
    if n_bad_scores > 0:
        print(f"  WARNING: {n_bad_scores} rows had a non-numeric {BAT_SCORE_COL} and will be treated as 0")
        df[BAT_SCORE_COL] = df[BAT_SCORE_COL].fillna(0)

    daily = df.groupby("_day").agg(
        n_posts_total=(BAT_SCORE_COL, "size"),
        n_burnout=(BAT_SCORE_COL, lambda s: (s >= 1).sum()),
    )

    for col in present_constructs:
        df[f"_{col}_flag"] = to_positive_flag(df[col])
        daily[f"n_{col.upper()}"] = df.groupby("_day")[f"_{col}_flag"].sum()

    # fill any calendar days missing from the raw data with 0 -- these are
    # days with no burnout-flagged posts, not necessarily collection gaps, but
    # worth a manual sanity check if you see long unbroken runs of zeros.
    full_range = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
    daily = daily.reindex(full_range, fill_value=0)
    daily.index.name = "date"
    daily = daily.reset_index()

    out_path = os.path.join(OUT_DIR, "daily_burnout_series.csv")
    daily.to_csv(out_path, index=False)
    print(f"\nSaved -> {out_path}")
    print(f"Date range: {daily['date'].min().date()} to {daily['date'].max().date()} ({len(daily)} days)")
    zero_run_check = (daily.drop(columns="date") == 0).all(axis=1).sum()
    if zero_run_check > 0:
        print(f"  NOTE: {zero_run_check} days have all-zero counts across every metric -- "
              f"worth spot-checking these aren't a data collection gap.")


if __name__ == "__main__":
    main()

Loading /Users/nadia/Desktop/redditRun_june/bat_posts_results_with_dates_v2.csv ...
Loaded 144652 posts
Columns found: ['row_type', 'post_id', 'comment_id', 'text', 'triage', 'na_subtype', 'triage_reason', 'EX', 'EMO', 'COG', 'MD', 'bat_score', 'EX_reasoning', 'EMO_reasoning', 'COG_reasoning', 'MD_reasoning', 'created_utc', 'subreddit', 'created_date', 'year', 'month', 'day_of_week', 'hour', 'year_month']

row_type value counts:
row_type
post    144652
ROW_TYPE_FILTER is None -- including ALL rows (posts + comments) as-is.
If that's not what you want, set ROW_TYPE_FILTER above to the value that
means 'post' and rerun.
  Found construct columns: ['EX', 'EMO', 'COG', 'MD']
  Sample values in 'EX': ['NO' 'YES']

Saved -> /Users/nadia/Desktop/redditRun_june/event_study_v2/daily_burnout_series.csv
Date range: 2018-01-01 to 2026-04-25 (3037 days)
